# Usage workload driver (admin)

Creates realistic **usage activity** in the quickstart's Coco content, so the measures built on it have
something to show: each element's `UsageActivity` (counted by Egeria's Pepys Activity Observer) and each
collection's `MemberPopularityProfile` (built from those counts by the Lovelace popularity profiler).

Several Coco personas each log in, search for the things they care about, open some of what they find
(a few elements far more often than others), look inside their favourite collections, and leave one
like and one rating on the element they used most.

* **Reads, plus a little feedback.** It creates no catalog elements. Each persona adds at most one like
  and one rating per run; a rating it already gave is replaced, not duplicated.
* **Safe to re-run**, and meant to be: every run adds more usage. It is bounded by `TIME_BUDGET_SECONDS`.
* **Results are not immediate.** Pepys records activity on its own refresh, and the popularity profiler
  runs when the Babbage Analytical Engine refreshes, so profiles change some minutes after a run.

This lives with the portal's own initialization files, not in `coco-workbooks`, and is not selected by
default in **Admin → Data Initialization**.

In [ ]:
# Parameters
PERSONAS = {
    # userid: search strings for the things this persona looks for
    "erinoverview":   ["Data Hub", "Product", "Supply Chain"],
    "peterprofile":   ["Customer", "Sales", "System"],
    "calliequartile": ["Clinical", "Trial", "Patient"],
    "faithbroker":    ["Privacy", "Personal", "Consent"],
    "harryhopeful":   ["Clinical Trial", "Treatment", "Hospital"],
    "pollytasker":    ["Project", "Governance", "Sustainability"],
}
GETS_PER_PERSONA = 12         # opens drawn from what the persona found, popular ones more often
COLLECTIONS_TO_EXPLORE = 2    # of the persona's most-opened collections, look at members
MEMBER_GETS = 4               # member opens per explored collection
PAGE_SIZE = 25                # results per search
FEEDBACK = True               # one like and one rating per persona
TIME_BUDGET_SECONDS = 600     # stop starting new work after this; the batch allows 900 per file
SEED = None                   # set an int to repeat a run's choices exactly

In [ ]:
import os, random, time
from collections import Counter, defaultdict
from pyegeria import CollectionManager, GlossaryManager, ProductManager, ClassificationExplorer

VIEW_SERVER = os.environ.get("EGERIA_VIEW_SERVER", "qs-view-server")
PLATFORM_URL = os.environ.get("EGERIA_VIEW_SERVER_URL") or os.environ.get("EGERIA_PLATFORM_URL", "https://localhost:9443")
seed = SEED if SEED is not None else random.randrange(1_000_000)
rng = random.Random(seed)
deadline = time.monotonic() + TIME_BUDGET_SECONDS
print(f"view server {VIEW_SERVER} at {PLATFORM_URL}; seed {seed}")


def password_for(userid):
    # Same rule as bootstrap_batches._password_for.
    return (os.environ.get(f"EGERIA_BOOTSTRAP_PASSWORD_{userid.upper()}")
            or os.environ.get("EGERIA_USER_PASSWORD") or "secret")


class OutOfTime(Exception):
    pass


stats = defaultdict(Counter)      # persona -> counts by kind
opened = Counter()                # (guid, name) -> opens this run
problems = []                     # (persona, what, message)
feedback_given = []               # (persona, guid, name, stars)


def call(persona, kind, fn, *args, **kwargs):
    if time.monotonic() > deadline:
        raise OutOfTime()
    try:
        result = fn(*args, **kwargs)
        stats[persona][kind] += 1
        return result
    except Exception as exc:
        stats[persona][kind + " failed"] += 1
        problems.append((persona, kind, str(exc).strip().splitlines()[0][:160] if str(exc).strip() else type(exc).__name__))
        return None


def items(result):
    return [r for r in result if isinstance(r, dict)] if isinstance(result, list) else []


def ident(el):
    hdr = el.get("elementHeader") or {}
    props = el.get("properties") or {}
    return hdr.get("guid"), props.get("displayName") or props.get("qualifiedName") or hdr.get("guid")


def weighted(pool, k):
    # Draw k with replacement, earlier (better-matching) results far more often.
    if not pool:
        return []
    weights = [1 / (rank + 1) for rank in range(len(pool))]
    return rng.choices(pool, weights=weights, k=k)

In [ ]:
def run_persona(userid, interests):
    pwd = password_for(userid)
    coll = CollectionManager(VIEW_SERVER, PLATFORM_URL, userid, pwd)
    token = call(userid, "login", coll.create_egeria_bearer_token)
    if not token:
        return
    gloss = GlossaryManager(VIEW_SERVER, PLATFORM_URL, userid, pwd)
    prod = ProductManager(VIEW_SERVER, PLATFORM_URL, userid, pwd)
    expl = ClassificationExplorer(VIEW_SERVER, PLATFORM_URL, userid, pwd)
    for c in (gloss, prod, expl):
        c.set_bearer_token(token)

    found = {"collection": [], "term": [], "product": []}
    for s in interests:
        found["collection"] += items(call(userid, "find", coll.find_collections, s, starts_with=False,
                                          output_format="JSON", page_size=PAGE_SIZE))
        found["term"] += items(call(userid, "find", gloss.find_glossary_terms, s, starts_with=False,
                                    output_format="JSON", page_size=PAGE_SIZE))
        found["product"] += items(call(userid, "find", prod.find_digital_products, s, starts_with=False,
                                       ignore_case=True, output_format="JSON", page_size=PAGE_SIZE))
    stats[userid]["elements found"] += sum(len(v) for v in found.values())

    getters = {"collection": coll.get_collection_by_guid, "term": gloss.get_term_by_guid,
               "product": prod.get_digital_product_by_guid}
    pool = [(kind, ident(el)) for kind in found for el in found[kind] if ident(el)[0]]
    mine = Counter()
    for kind, (guid, name) in weighted(pool, GETS_PER_PERSONA):
        if call(userid, "open", getters[kind], guid) is not None:
            opened[(guid, name)] += 1
            mine[(kind, guid, name)] += 1

    top_collections = [k for k in mine if k[0] == "collection"][:COLLECTIONS_TO_EXPLORE]
    for _, guid, _name in top_collections:
        members = items(call(userid, "list members", coll.get_collection_members, collection_guid=guid,
                             output_format="JSON"))
        member_pool = [ident(m) for m in members if ident(m)[0]]
        for mguid, mname in weighted(member_pool, MEMBER_GETS):
            got = call(userid, "open member", expl.get_element_by_guid, guid=mguid, output_format="JSON",
                       body={"class": "GetRequestBody", "graphQueryDepth": 0})
            if got is not None:
                opened[(mguid, mname)] += 1

    if FEEDBACK and mine:
        (_, fguid, fname), _n = mine.most_common(1)[0]
        likes = items(call(userid, "read likes", expl.get_attached_likes, fguid))
        if not any(((l.get("elementHeader") or {}).get("versions") or {}).get("createdBy") == userid for l in likes):
            call(userid, "like", expl.add_like_to_element, fguid)   # one like per user; keep an existing one
        try:
            expl.remove_rating_from_element(fguid)   # one rating per user: replace, don't add
        except Exception:
            pass
        stars = 3 + (sum(map(ord, userid + fguid)) % 3)
        rated = call(userid, "rating", expl.add_rating_to_element, fguid, is_public=True,
                     body={"class": "NewAttachmentRequestBody",
                           "properties": {"class": "RatingProperties", "starRating": stars}})
        feedback_given.append((userid, fguid, fname, stars if rated is not None else None))


started = time.monotonic()
for userid, interests in PERSONAS.items():
    try:
        run_persona(userid, interests)
    except OutOfTime:
        print(f"time budget reached during {userid}; stopping")
        break
print(f"done in {time.monotonic() - started:.0f}s")

In [ ]:
kinds = ["login", "find", "elements found", "open", "list members", "open member", "like", "rating"]
print(f"{'persona':16}" + "".join(f"{k:>15}" for k in kinds) + f"{'failed':>8}")
for userid in PERSONAS:
    s = stats.get(userid, Counter())
    failed = sum(v for k, v in s.items() if k.endswith(" failed"))
    print(f"{userid:16}" + "".join(f"{s.get(k, 0):>15}" for k in kinds) + f"{failed:>8}")

print("\nMost opened this run:")
for (guid, name), n in opened.most_common(10):
    print(f"{n:4}  {name}  ({guid})")

if feedback_given:
    print("\nLiked and rated (feedback elements attached to these):")
    for persona, guid, name, stars in feedback_given:
        print(f"  {persona:16} {('%d stars' % stars) if stars else 'rating failed':14} {name}  ({guid})")

if problems:
    print(f"\n{len(problems)} failed calls (first 10):")
    for persona, what, msg in problems[:10]:
        print(f"  {persona:16} {what:12} {msg}")